## Étape 5 — Recherche

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

client = chromadb.PersistentClient(path="../base_vectorielle")
collection = client.get_collection("altea")
print("Chunks en base :", collection.count())      

c:\Users\antoi\Desktop\antoine_cfa\cours M2\TP_rag_Altea_Antoine_Brigouleix\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2215.45it/s]


Chunks en base : 69


## Q1 : la fonction rechercher

In [2]:
def rechercher(question: str, k: int = 3, service: str | None = None) -> list[dict]:
    """
    Renvoie les k chunks les plus proches de la question, avec leur source, page, service et score.
    """
    vecteur = model.encode([question], normalize_embeddings=True)
    res = collection.query(
        query_embeddings=vecteur.tolist(),
        n_results=k,
        where={"service": service} if service else None,
    )
    return [
        {
            "texte": doc,
            "source": meta["source"],
            "page": meta["page"],
            "service": meta["service"],
            "score": 1 - dist,     
        }
        for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0])
    ]


def afficher(resultats: list[dict], apercu: int = 120):
    for r in resultats:
        print(f"  {r['score']:.3f} | {r['service']:7} | {r['source']} p.{r['page']}")
        print("     ", r["texte"][:apercu].replace("\n", " "), "...")

In [7]:
questions = [
    "Combien de jours de télétravail par semaine ?",
    "J'ai oublié mon mot de passe, que faire ?",
    "À quelle heure ouvre le restaurant d'entreprise ?",
]
for q in questions:
    print(f"\n❓ {q}")
    afficher(rechercher(q))


❓ Combien de jours de télétravail par semaine ?
  0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de leur tuteur. 2. Nombr ...
  0.615 | RH      | conges_absences.pdf p.1
      de congés payés par mois de travail effectif, soit 30 jours ouvrables (5 semaines) pour une année complète. La période d ...
  0.575 | RH      | conges_absences.pdf p.1
      oins 2 semaines à l'avance pour une absence de 1 à 3 jours, et au moins 1 mois à l'avance pour une absence de plus d'une ...

❓ J'ai oublié mon mot de passe, que faire ?
  0.698 | IT      | mot_de_passe.txt p.1
      MOT DE PASSE OUBLIÉ OU EXPIRÉ Altéa Services — Support IT — Référence IT-003  Vous avez oublié votre mot de passe, ou vo ...
  0.591 | IT      | charte_informatique.pdf p.1
      utilisateur dispose d'un compte personnel. Il ne doit jamais communiquer son mot de passe, y compris au support informat ...
  0.509 | IT      | securite_email

## Q2 : effet de k 

In [4]:
q = "Combien de jours de télétravail par semaine ?"
for k in [1, 3, 10]:
    print(f"\n===== k = {k} =====")
    afficher(rechercher(q, k=k), apercu=60)


===== k = 1 =====
  0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravaille ...

===== k = 3 =====
  0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravaille ...
  0.615 | RH      | conges_absences.pdf p.1
      de congés payés par mois de travail effectif, soit 30 jours  ...
  0.575 | RH      | conges_absences.pdf p.1
      oins 2 semaines à l'avance pour une absence de 1 à 3 jours,  ...

===== k = 10 =====
  0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravaille ...
  0.615 | RH      | conges_absences.pdf p.1
      de congés payés par mois de travail effectif, soit 30 jours  ...
  0.575 | RH      | conges_absences.pdf p.1
      oins 2 semaines à l'avance pour une absence de 1 à 3 jours,  ...
  0.548 | RH      | teletravail.pdf p.1
      ossible le mardi. À titre exceptionnel (grève des transports ...
  0.506 | General | horaires.txt p.1
 

## Q3 : filtre par service 

In [8]:
q = "Combien de jours de télétravail par semaine ?"
for service in ["RH", "IT"]:
    print(f"\n===== service = {service} =====")
    afficher(rechercher(q, k=3, service=service), apercu=60)


===== service = RH =====
  0.709 | RH      | teletravail.pdf p.1
      bles. Les alternants et les stagiaires peuvent télétravaille ...
  0.615 | RH      | conges_absences.pdf p.1
      de congés payés par mois de travail effectif, soit 30 jours  ...
  0.575 | RH      | conges_absences.pdf p.1
      oins 2 semaines à l'avance pour une absence de 1 à 3 jours,  ...

===== service = IT =====
  0.425 | IT      | demande_materiel.txt p.1
      l'équipement dans le catalogue (second écran, station d'accu ...
  0.421 | IT      | securite_emails.txt p.1
      ir besoin d'analyser l'ordinateur.  4. Demandes de virement  ...
  0.416 | IT      | demande_materiel.txt p.1
      mois avant pour planifier l'échange et la copie de vos donné ...


## Q4 : la fonction construire_prompt

In [25]:
def construire_prompt(question: str, resultats: list[dict]) -> str:
    extraits = "\n\n".join(
        f"[{i}] (source : {r['source']}, page {r['page']})\n{r['texte']}"
        for i, r in enumerate(resultats, start=1)
    )
    return f"""Tu es l'assistant de documentation interne d'Altéa Services.
Réponds uniquement à partir des extraits ci-dessous, en français.
Indique à la fin les sources utilisées (nom du fichier et page).
Si les extraits ne contiennent pas la réponse, réponds exactement : "Je n'ai pas trouvé cette information dans la documentation d'Altéa."

EXTRAITS :
{extraits}

QUESTION : {question}

RÉPONSE :"""


question = "Combien de jours de télétravail par semaine ?"
prompt = construire_prompt(question, rechercher(question, k=3))
print(prompt)

Tu es l'assistant de documentation interne d'Altéa Services.
Réponds uniquement à partir des extraits ci-dessous, en français.
Indique à la fin les sources utilisées (nom du fichier et page).
Si les extraits ne contiennent pas la réponse, réponds exactement : "Je n'ai pas trouvé cette information dans la documentation d'Altéa."

EXTRAITS :
[1] (source : teletravail.pdf, page 1)
bles.
Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de
leur tuteur.
2. Nombre de jours autorisés
Chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. Ces jours sont fixés
avec le manager ; ils ne peuvent pas être reportés d'une semaine sur l'autre.
Le mardi est un jour de présence obligatoire pour tous les salariés : c'est le jour des réunions
d'équipe. Aucun télétravail n'est donc possible le mardi.
À titre exceptionnel (grève des 

[2] (source : conges_absences.pdf, page 1)
de congés payés par mois de travail effectif, soit
30 jours ouvrables (5 s

In [18]:
resultats = rechercher("Combien de jours de télétravail par semaine ?", k=10)
for rang, r in enumerate(resultats, start=1):
    if "par semaine" in r["texte"] or "jours" in r["texte"]:
        print(f"rang {rang} | {r['source']} p.{r['page']} | score {r['score']:.3f}")
        print(r["texte"], "\n" + "-" * 60)

rang 1 | teletravail.pdf p.1 | score 0.709
bles.
Les alternants et les stagiaires peuvent télétravailler 1 jour par semaine, avec l'accord de
leur tuteur.
2. Nombre de jours autorisés
Chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. Ces jours sont fixés
avec le manager ; ils ne peuvent pas être reportés d'une semaine sur l'autre.
Le mardi est un jour de présence obligatoire pour tous les salariés : c'est le jour des réunions
d'équipe. Aucun télétravail n'est donc possible le mardi.
À titre exceptionnel (grève des  
------------------------------------------------------------
rang 2 | conges_absences.pdf p.1 | score 0.615
de congés payés par mois de travail effectif, soit
30 jours ouvrables (5 semaines) pour une année complète. La période d'acquisition court du
1er juin au 31 mai.
Les salariés au forfait jours bénéficient en plus de 10 jours de RTT par an. Les salariés à 35
heures hebdomadaires n'ont pas de RTT.
Deux jours de congés supplémentaires, dits jours d'

## 6 -- Génération 

In [26]:
import ollama

MODELE = "llama3.2"
SEUIL = 0.49      

def generer(prompt: str, modele: str = MODELE) -> str:
    reponse = ollama.chat(
        model=modele,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0},
    )
    return reponse["message"]["content"]

REPLI = "Je n'ai pas trouvé cette information dans la documentation d'Altéa."

def repondre(question: str, k: int = 3, service: str | None = None):
    resultats = rechercher(question, k=k, service=service)
    if not resultats or resultats[0]["score"] < SEUIL:
        return REPLI, resultats                      # inutile d'appeler le LLM
    return generer(construire_prompt(question, resultats)), resultats

## test ollama

In [27]:
reponse, sources = repondre("Combien de jours de télétravail par semaine ?")
print(reponse)
print("\nSources :", [(s["source"], s["page"], round(s["score"], 3)) for s in sources])

Selon les extraits fournis, chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. Cependant, il est important de noter que ces jours sont fixés avec le manager et ne peuvent pas être reportés d'une semaine sur l'autre.

Source : [1] (teletravail.pdf, page 1)

Sources : [('teletravail.pdf', 1, 0.709), ('conges_absences.pdf', 1, 0.615), ('conges_absences.pdf', 1, 0.575)]


## Q5 : les quesuions 

In [28]:
questions = [
    "Combien de jours de télétravail par semaine ?",
    "J'ai oublié mon mot de passe, que faire ?",
    "À quelle heure ouvre le restaurant d'entreprise à Paris ?",
    "Quel temps fera-t-il demain ?",                        
    "Quel est le salaire du directeur général ?",           
]
for q in questions:
    reponse, sources = repondre(q)
    print(f"\n❓ {q}")
    print(f"💬 {reponse}")
    print("   Sources :", [(s["source"], s["page"], round(s["score"], 2)) for s in sources[:3]])


❓ Combien de jours de télétravail par semaine ?
💬 Selon les extraits fournis, chaque salarié éligible peut télétravailler jusqu'à 2 jours par semaine. Cependant, il est important de noter que ces jours sont fixés avec le manager et ne peuvent pas être reportés d'une semaine sur l'autre.

Source : [1] (teletravail.pdf, page 1)
   Sources : [('teletravail.pdf', 1, 0.71), ('conges_absences.pdf', 1, 0.61), ('conges_absences.pdf', 1, 0.57)]

❓ J'ai oublié mon mot de passe, que faire ?
💬 Pour oublier son mot de passe, il faut suivre les étapes suivantes :

1. Rendez-vous sur la page de réinitialisation, accessible depuis l'écran de connexion de votre ordinateur (lien « Mot de passe oublié ? ») ou depuis votre téléphone sur le portail compte.altea-services.example.
2. Saisissez votre identifiant (prénom.nom), puis validez le code reçu par SMS.

Source : [1] (source : mot_de_passe.txt, page 1)

Je n'ai pas trouvé cette information dans la documentation d'Altéa.
   Sources : [('mot_de_passe.tx

## Q6 

In [29]:
print(generer("Combien de jours de télétravail par semaine sont autorisés chez Altéa ?"))

Je ne peux pas fournir d'informations sur les politiques spécifiques d'Altéa, car je n'ai pas accès à cette information.


## Q7 Pourquoi mettre temperature = 0 ?
  Elle rend la réponse la plus déterministe possible : même question, même réponse. Pour un assistant documentaire on veut de la fidélité aux sources, pas de la créativité.

## Q8 le fichier evaluation.csv

In [31]:
import pandas as pd
from pathlib import Path

evaluation = pd.DataFrame([
    ("Combien de jours de télétravail par semaine un salarié peut-il faire ?", "teletravail.pdf"),
    ("Peut-on télétravailler le mardi ?",                                      "teletravail.pdf"),
    ("Combien de jours de congés payés acquiert-on sur une année complète ?",  "conges_absences.pdf"),
    ("Combien de jours de RTT pour un salarié au forfait jours ?",             "conges_absences.pdf"),
    ("À quelle heure ouvre le restaurant d'entreprise ?",                      "restauration.txt"),
    ("Quels sont les horaires d'ouverture du site de Lyon ?",                  "horaires.txt"),
    ("Comment réinitialiser mon mot de passe ?",                               "mot_de_passe.txt"),
    ("Comment demander du matériel informatique ?",                            "demande_materiel.txt"),   # à vérifier
    ("Comment me connecter au VPN ?",                                          "vpn.txt"),                # à vérifier
    ("Comment se faire rembourser une note de frais ?",                        "notes_de_frais.pdf"),     # à vérifier
], columns=["question", "document_attendu"])

Path("../data").mkdir(exist_ok=True)
evaluation.to_csv("../data/evaluation.csv", index=False, encoding="utf-8")
evaluation

,question,document_attendu
0,Combien de jours de télétravail par semaine un...,teletravail.pdf
1,Peut-on télétravailler le mardi ?,teletravail.pdf
2,Combien de jours de congés payés acquiert-on s...,conges_absences.pdf
3,Combien de jours de RTT pour un salarié au for...,conges_absences.pdf
4,À quelle heure ouvre le restaurant d'entreprise ?,restauration.txt
5,Quels sont les horaires d'ouverture du site de...,horaires.txt
6,Comment réinitialiser mon mot de passe ?,mot_de_passe.txt
7,Comment demander du matériel informatique ?,demande_materiel.txt
8,Comment me connecter au VPN ?,vpn.txt
9,Comment se faire rembourser une note de frais ?,notes_de_frais.pdf


In [32]:
for r in rechercher("Comment me connecter au VPN ?", k=3):
    print(r["source"], round(r["score"], 3), "|", r["texte"][:200].replace("\n", " "))

vpn.txt 0.755 | hone (authentification à deux facteurs). L'icône devient verte une fois la connexion établie.  2. Je n'arrive pas à me connecter au VPN : que faire ? - Vérifiez d'abord que votre connexion internet fo
vpn.txt 0.738 | CONNEXION AU VPN Altéa Services — Support IT — Référence IT-005  Le VPN est obligatoire pour accéder aux applications internes (MonEspace, Frais+, espace partagé) en dehors des locaux, notamment en té
vpn.txt 0.6 | a connexion de votre téléphone professionnel. - Redémarrez l'ordinateur, puis relancez AltéaConnect. - Si le message « Certificat expiré » s'affiche, votre ordinateur doit être reconnecté au réseau de


## Q9 : taux de réussite de la recherche

In [33]:
def evaluer_recherche(k: int = 3) -> pd.DataFrame:
    lignes = []
    for _, row in pd.read_csv("../data/evaluation.csv").iterrows():
        resultats = rechercher(row["question"], k=k)
        sources = [r["source"] for r in resultats]
        rang = sources.index(row["document_attendu"]) + 1 if row["document_attendu"] in sources else None
        lignes.append({
            "question": row["question"][:55],
            "attendu": row["document_attendu"],
            "trouvé": rang is not None,
            "rang": rang,
            "score_top1": round(resultats[0]["score"], 3),
        })
    return pd.DataFrame(lignes)

df_rech = evaluer_recherche(k=3)
print(df_rech.to_string(index=False))
print(f"\nTaux de réussite de la recherche (k=3) : {df_rech['trouvé'].mean():.0%}")

                                               question              attendu  trouvé  rang  score_top1
Combien de jours de télétravail par semaine un salarié       teletravail.pdf    True     1       0.810
                      Peut-on télétravailler le mardi ?      teletravail.pdf    True     1       0.657
Combien de jours de congés payés acquiert-on sur une an  conges_absences.pdf    True     1       0.714
Combien de jours de RTT pour un salarié au forfait jour  conges_absences.pdf    True     1       0.900
      À quelle heure ouvre le restaurant d'entreprise ?     restauration.txt    True     2       0.530
  Quels sont les horaires d'ouverture du site de Lyon ?         horaires.txt    True     1       0.782
               Comment réinitialiser mon mot de passe ?     mot_de_passe.txt    True     1       0.678
            Comment demander du matériel informatique ? demande_materiel.txt    True     1       0.706
                          Comment me connecter au VPN ?              vpn.

In [37]:
  mrr = (1 / df_rech["rang"]).mean()
  print(f"MRR : {mrr:.3f}")      

MRR : 0.950


## Q10: générer les 10 réponses et les noter

In [41]:
lignes = []
for _, row in pd.read_csv("../data/evaluation.csv").iterrows():
    reponse, resultats = repondre(row["question"])
    lignes.append({
        "question": row["question"],
        "document_attendu": row["document_attendu"],
        "reponse": reponse,
        "document_cite": row["document_attendu"] in reponse,   
        "note": "",          
        "sources_ok": "",    
    })

resultats_gen = pd.DataFrame(lignes)
resultats_gen.to_csv("../data/resultats_generation.csv", index=False, encoding="utf-8-sig")

for l in lignes:
    print(f"\n❓ {l['question']}\n💬 {l['reponse']}\n   (fichier attendu cité : {l['document_cite']})")


❓ Combien de jours de télétravail par semaine un salarié peut-il faire ?
💬 Un salarié éligible peut télétravailler jusqu'à 2 jours par semaine.

Source : [1] (teletravail.pdf, page 1)
   (fichier attendu cité : True)

❓ Peut-on télétravailler le mardi ?
💬 Non, il est interdit de télétravailler le mardi, car ce jour est désigné comme jour de présence obligatoire pour tous les salariés et aucun télétravail n'est possible le mardi.

Source : teletravail.pdf, page 1.
   (fichier attendu cité : True)

❓ Combien de jours de congés payés acquiert-on sur une année complète ?
💬 Selon les extraits fournis, un salarié acquiert 30 jours ouvrables (5 semaines) de congés payés par an en cas d'année complète.

Source : conges_absences.pdf, page 1.
   (fichier attendu cité : True)

❓ Combien de jours de RTT pour un salarié au forfait jours ?
💬 Selon l'extrait [1] (conges_absences.pdf, page 1), les salariés au forfait jours bénéficient en plus de 10 jours de RTT par an.

La réponse est donc de 10 jour

In [42]:
notes = pd.read_csv("../data/resultats_generation.csv", encoding="utf-8-sig")
print("Réponses justes     :", f"{(notes['note'] == 'juste').mean():.0%}")
print("Réponses sourcées OK:", f"{(notes['sources_ok'] == 'oui').mean():.0%}")

Réponses justes     : 0%
Réponses sourcées OK: 0%


## Q11 

In [43]:
q = "Comment se faire rembourser une note de frais ?"
for r in rechercher(q, k=3):
    print(f"{r['source']} p.{r['page']} | {r['score']:.3f}\n{r['texte']}\n{'-' * 60}")

res = collection.get(where={"source": "notes_de_frais.pdf"})
for d in res["documents"]:
    if "jours" in d:
        print(d, "\n" + "=" * 60)

notes_de_frais.pdf p.2 | 0.718
alidation par le manager puis par la comptabilité, le remboursement est versé par
virement sous 10 jours ouvrés environ.
5. Dépenses exclues
Ne sont pas remboursés : les boissons alcoolisées hors repas d'affaires, les amendes et
contraventions, les dépenses personnelles (minibar, films payants), les pourboires au-delà de
10 % de l'addition.
------------------------------------------------------------
notes_de_frais.pdf p.1 | 0.538
Altéa Services — Document interne — Diffusion restreinte
1
Document fictif réalisé à des fins pédagogiques
Notes de frais
RH-007 — version 3 — en vigueur au 1er mars 2026
1. Principe
Les frais engagés pour le compte de l'entreprise (déplacements, repas d'affaires,
hébergement) sont remboursés sur présentation d'une note de frais accompagnée des
justificatifs originaux.
2. Plafonds de remboursement
Type de dépense
Plafond
Repas du midi en déplacement
20 € par repas
Repas du soir en déplacement
30
---------------------------------

In [39]:
for i in range(4, 9):          # les questions 5 à 9, celles qui ont été coupées
    print(f"\n❓ {resultats_gen.loc[i, 'question']}\n💬 {resultats_gen.loc[i, 'reponse']}")


❓ À quelle heure ouvre le restaurant d'entreprise ?
💬 Le restaurant d'entreprise est ouvert de 11 h 45 à 14 h.

(Sources : [2] - Restaurant d'entreprise (Paris et Lyon), page 1)

❓ Quels sont les horaires d'ouverture du site de Lyon ?
💬 Les sites de Paris et de Lyon sont ouverts du lundi au vendredi de 7 h 30 à 19 h. Accueil des visiteurs de 9 h à 17 h 30.

(Sources : horaires.txt, page 1)

❓ Comment réinitialiser mon mot de passe ?
💬 Pour réinitialiser votre mot de passe, rendez-vous sur la page de réinitialisation, accessible depuis l'écran de connexion de votre ordinateur (lien « Mot de passe oublié ? ») ou depuis votre téléphone sur le portail compte.altea-services.example. Saisissez votre identifiant (prénom.nom), puis validez le code reçu par SMS sur votre téléphone professionnel ou dans l'application d'authentification.

Source : mot_de_passe.txt, page 1

❓ Comment demander du matériel informatique ?
💬 Pour demander du matériel informatique, il faut suivre les étapes suivantes 

In [45]:
def construire_prompt(question: str, resultats: list[dict]) -> str:
    extraits = "\n\n".join(
        f"[{i}] (source : {r['source']}, page {r['page']})\n{r['texte']}"
        for i, r in enumerate(resultats, start=1)
    )
    return f"""Tu es l'assistant de documentation interne d'Altéa Services.
Réponds en français, uniquement à partir des extraits ci-dessous.

Règles :
1. Si un extrait contient la réponse, donne-la de façon complète et précise, sans ajouter d'autre commentaire.
2. Pour un délai, un montant ou un nombre, reprends-le tel qu'il est écrit dans l'extrait et précise exactement à quoi il s'applique (qui fait quoi, et à quel moment). Ne rattache jamais un chiffre à une autre étape que celle de l'extrait.
3. Termine par une ligne « Sources : » avec le nom du fichier et la page des extraits utilisés.
4. Si, et seulement si, aucun extrait ne contient la réponse, réponds uniquement : "{REPLI}" et rien d'autre.

EXTRAITS :
{extraits}

QUESTION : {question}

RÉPONSE :"""

In [46]:
q = "Comment se faire rembourser une note de frais ?"
reponse, _ = repondre(q)
print(reponse)

Pour obtenir le remboursement d'une note de frais, il faut suivre les étapes suivantes :

1. L'employé doit présenter la note de frais accompagnée des justificatifs originaux au service RH.
2. La note de frais est ensuite validée par le manager et par la comptabilité.
3. Une fois validée, le remboursement est effectué par virement sous 10 jours ouvrés environ.

Sources : notes_de_frais.pdf, page 1.


In [47]:
lignes = []
for _, row in pd.read_csv("../data/evaluation.csv").iterrows():
    reponse, resultats = repondre(row["question"])
    lignes.append({"question": row["question"], "reponse_v2": reponse,
                   "document_cite": row["document_attendu"] in reponse})
v2 = pd.DataFrame(lignes)
v2.to_csv("../data/resultats_generation_v2.csv", index=False, encoding="utf-8-sig")
print("Fichier attendu cité :", f"{v2['document_cite'].mean():.0%}")

Fichier attendu cité : 100%


In [ ]:
source = "teletravail.pdf"
anciens = [c for c in chunks if c["source"] == source]
print(len(anciens), "chunks pour", source)

avant = collection.count()
reindexer_document(source, anciens[:-1])     
print("Après modification :", avant, "->", collection.count())   

reindexer_document(source, anciens)          
print("Après restauration :", collection.count())              

7 chunks pour teletravail.pdf
Après modification : 69 -> 68
Après restauration : 69


## Q12 : cf image "Q12.png"

## Q14 :
Droits d’accès : le filtre service est choisi par l’utilisateur. Il faut authentifier les salariés et appliquer leurs droits côté serveur.

Données personnelles : les documents peuvent contenir des noms ou des données RH ; il faut les identifier (RGPD) et les exclure ou les anonymiser avant l’indexation.

Mise à jour des documents : un document modifié doit être réindexé (supprimer puis réinsérer, comme reindexer_document), sinon l’assistant répond avec une règle périmée.

Coût et hébergement du LLM : un modèle local demande des serveurs adaptés (GPU), un modèle hébergé facture à l’usage et envoie les questions à un tiers ; il faut aussi suivre la qualité dans la durée.